# Seleção de variáveis com Lasso

**Autor:** Prof. Dr. Sergio Antônio Andrade de Freitas  
**Material da disciplina:** FGA0083 — Aprendizado de Máquina, UnB

## Objetivo e dados

Recuperar as variáveis realmente associadas ao alvo sintético.

**Pergunta-guia:** Quais coeficientes o Lasso reduz a zero e como isso muda com alpha?

Execute as células na ordem e interprete os resultados antes de fazer o exercício.

In [1]:
import numpy as np
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(0)
X = rng.normal(size=(300, 10))
# Somente três das dez variáveis geram o alvo contínuo.
y = 3 * X[:, 0] - 2 * X[:, 3] + 1.5 * X[:, 7] + rng.normal(scale=0.5, size=300)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
resultados_cv = {}
for alpha in (0.01, 0.1, 1.0):
    modelo = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))
    rmse_cv = -cross_val_score(modelo, X_train, y_train, cv=5,
                              scoring='neg_root_mean_squared_error').mean()
    resultados_cv[alpha] = rmse_cv
    print(f'alpha={alpha}: RMSE validação={rmse_cv:.3f}')
melhor_alpha = min(resultados_cv, key=resultados_cv.get)
modelo_final = make_pipeline(StandardScaler(), Lasso(alpha=melhor_alpha, max_iter=10000))
modelo_final.fit(X_train, y_train)
rmse_teste = np.sqrt(mean_squared_error(y_test, modelo_final.predict(X_test)))
print(f'alpha escolhido={melhor_alpha}; RMSE no teste reservado={rmse_teste:.3f}')
print('Coeficientes:', np.round(modelo_final[-1].coef_, 2))

alpha=0.01: RMSE validação=0.540
alpha=0.1: RMSE validação=0.556
alpha=1.0: RMSE validação=1.804
alpha escolhido=0.01; RMSE no teste reservado=0.585
Coeficientes: [ 3.04 -0.   -0.04 -1.99 -0.03 -0.    0.01  1.43  0.01  0.01]


## Interpretação e limites

O alvo é sintético e contínuo; não representa adesão clínica.

## Exercício

Compare alpha em validação e tente `data/6.1 - dados_elastic_net_regression.csv` como extensão.

Registre a hipótese, a mudança realizada, a métrica ou figura observada e uma conclusão que os dados de fato sustentem.